In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Mestrado/Dissertacao/app"
!pip install -r requirements.txt

/content/drive/MyDrive/Mestrado/Dissertacao/app
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.

In [1]:
import sys

sys.path.append("/content/drive/MyDrive/Mestrado/Dissertacao/app/multitask_unet")
sys.path.append("/content/drive/MyDrive/Mestrado/Dissertacao/app/dataset")

In [2]:
import torch
from torch import optim, nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import random
import os

from unet import UNet
from data_loader import CamusSequenceDataset

In [6]:
# Copia a pasta inteira do Drive para o ambiente local do Colab
!cp -r /content/drive/MyDrive/Mestrado/Dissertacao/app/dataset/train_valid /content/dataset_local



In [7]:
LEARNING_RATE = 3e-4
BATCH_SIZE = 32
EPOCHS = 50

MODEL_SAVE_PATH = "/content/drive/MyDrive/Mestrado/Dissertacao/results/unet.pth"

DATASET_PATH = "/content/dataset_local"

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)

Usando dispositivo: cuda


In [10]:
all_files = [f for f in os.listdir(DATASET_PATH) if "half_sequence.nii" in f and "_gt" not in f]
unique_patients = list(set([f.split('_')[0] for f in all_files]))
unique_patients.sort() # Ordena para garantir consistência antes de embaralhar

# 2. Embaralha a lista de pacientes (usamos uma seed para reprodutibilidade, se quiser)
random.seed(42)
random.shuffle(unique_patients)

# 3. Divide a lista (ex: 80% treino, 20% validação)
train_ratio = 0.8
split_idx = int(len(unique_patients) * train_ratio)

train_patients = unique_patients[:split_idx]
val_patients = unique_patients[split_idx:]

print(f"Total de pacientes na pasta: {len(unique_patients)}")
print(f"Pacientes para Treino: {len(train_patients)} | Validação: {len(val_patients)}")

# 4. Instancia os datasets passando as listas autorizadas
train_dataset = CamusSequenceDataset(data_dir=DATASET_PATH, allowed_patients=train_patients)
val_dataset = CamusSequenceDataset(data_dir=DATASET_PATH, allowed_patients=val_patients)

Total de pacientes na pasta: 425
Pacientes para Treino: 340 | Validação: 85
Dataset inicializado com 13079 frames totais.
Dataset inicializado com 3345 frames totais.


In [11]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
model = UNet(in_channels=1, num_classes=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

criterion_seg = nn.BCEWithLogitsLoss() # Para a máscara do ventrículo (1 canal)
criterion_cls = nn.CrossEntropyLoss()  # Para a classificação das câmaras (2 classes)


In [13]:
import os
os.makedirs("/content/drive/MyDrive/Mestrado/Dissertacao/results", exist_ok=True)

In [ ]:
for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_running_loss = 0
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    # Desempacotando img, mask e label
    for idx, (img, mask, label) in enumerate(pbar):
        img = img.to(device)
        mask = mask.to(device)
        label = label.to(device)

        # Forward (retorna predição espacial e predição de classe)
        pred_mask, pred_label = model(img)

        # Calculando as perdas individuais
        loss_seg = criterion_seg(pred_mask, mask)
        loss_cls = criterion_cls(pred_label, label)

        # Somando as perdas (você pode adicionar pesos aqui no futuro, ex: loss_seg + 0.5 * loss_cls)
        loss = loss_seg + loss_cls

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()
        pbar.set_postfix({"loss_total": loss.item(), "loss_seg": loss_seg.item(), "loss_cls": loss_cls.item()})

    train_loss = train_running_loss / len(train_dataloader)

    # Validação
    model.eval()
    val_running_loss = 0
    with torch.no_grad():
        for img, mask, label in val_dataloader:
            img = img.to(device)
            mask = mask.to(device)
            label = label.to(device)

            pred_mask, pred_label = model(img)

            loss_seg = criterion_seg(pred_mask, mask)
            loss_cls = criterion_cls(pred_label, label)
            loss = loss_seg + loss_cls

            val_running_loss += loss.item()

    val_loss = val_running_loss / len(val_dataloader)

    print(f"\nSummary Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print("-" * 30)

    torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("Treinamento concluído!")

Epoch 1/50: 100%|██████████| 409/409 [46:55<00:00,  6.88s/it, loss_total=0.347, loss_seg=0.0445, loss_cls=0.303]



Summary Epoch 1: Train Loss: 0.6485 | Val Loss: 0.1976
------------------------------


Epoch 2/50: 100%|██████████| 409/409 [47:51<00:00,  7.02s/it, loss_total=0.0417, loss_seg=0.0358, loss_cls=0.00587]



Summary Epoch 2: Train Loss: 0.1612 | Val Loss: 0.1315
------------------------------


Epoch 3/50: 100%|██████████| 409/409 [47:27<00:00,  6.96s/it, loss_total=0.0391, loss_seg=0.0351, loss_cls=0.00402]


...

Epoch 5/50: 100%|██████████| 409/409 [48:00<00:00,  7.04s/it, loss_total=0.0275, loss_seg=0.0274, loss_cls=0.000157]



Summary Epoch 5: Train Loss: 0.0466 | Val Loss: 0.1615
------------------------------


Epoch 6/50: 100%|██████████| 409/409 [48:01<00:00,  7.05s/it, loss_total=0.0341, loss_seg=0.0265, loss_cls=0.00754]



Summary Epoch 6: Train Loss: 0.0494 | Val Loss: 0.1005
------------------------------


Epoch 7/50:  39%|███▉      | 160/409 [18:55<30:00,  7.23s/it, loss_total=0.0325, loss_seg=0.0273, loss_cls=0.00522]